# Incremental Fine-tune PPE Model (New Datasets Only)

Notebook nay de train bo sung (fine-tune) cho model hien tai `best_ppe_model_v1.pt` voi dataset moi, khong dung lai 5 bo cu.

Muc tieu cai thien cac class dang yeu:
- helmet
- safety_harness
- boots
- trash_bag
- wet_floor_sign

Best practice: tiep tuc train tu model cu thay vi train lai tu dau de tiet kiem thoi gian.

In [1]:
!python -m pip install --upgrade pip -q
!pip install ultralytics roboflow python-dotenv ddgs requests pillow pyyaml -q

In [2]:
from pathlib import Path
import os
import re
import json
import yaml
import shutil
from urllib.parse import urlparse
from collections import defaultdict

import torch
from ultralytics import YOLO
from ddgs import DDGS

ROOT = Path.cwd()
DATA_ROOT = ROOT / 'new_data_sources'
MERGED_ROOT = ROOT / 'Master_Dataset_v2'
RUNS_ROOT = ROOT / 'runs_incremental'

for p in [DATA_ROOT, MERGED_ROOT, RUNS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('Workspace:', ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Workspace: e:\capstone\cleanopsai-ai
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## Class Targets

Muc tieu nhan (10 class):
- gloves
- surgical_mask
- face_shield
- biohazard_suit
- helmet
- safety_vest
- safety_harness
- boots
- trash_bag
- wet_floor_sign

In [3]:
TARGET_NAMES = {
    0: 'gloves',
    1: 'surgical_mask',
    2: 'face_shield',
    3: 'biohazard_suit',
    4: 'helmet',
    5: 'safety_vest',
    6: 'safety_harness',
    7: 'boots',
    8: 'trash_bag',
    9: 'wet_floor_sign',
}

FOCUS_CLASSES = {'helmet', 'safety_harness', 'boots', 'trash_bag', 'wet_floor_sign'}

OLD_DATASET_URLS = {
    'https://universe.roboflow.com/cppe-nxmvw/medical-ppe-o6ot6',
    'https://universe.roboflow.com/fypcv/safety-harness-iess4',
    'https://universe.roboflow.com/lena-f7w17/wet-floor-detection1',
    'https://universe.roboflow.com/new-workspace-slpmv/bag-qe348',
    'https://docs.ultralytics.com/datasets/detect/construction-ppe/',
}

## Tim Kiem Dataset Moi Tu Web

Cell nay tu dong search cac URL dataset moi lien quan PPE va loai bo URL cu.

Luu y:
- Ket qua search la goi y de ban review nhanh.
- Uu tien URL chua 'universe.roboflow.com' hoac tai lieu dataset detect uy tin.

In [4]:
queries = [
    'roboflow universe helmet ppe dataset',
    'roboflow universe safety harness dataset',
    'roboflow universe boots detection dataset',
    'roboflow universe trash bag detection dataset',
    'roboflow universe wet floor sign dataset',
    'construction ppe dataset yolo',
]

def normalize_url(u):
    u = str(u).strip()
    if u.endswith('/'):
        u = u[:-1]
    return u

def is_probably_rf_project(url: str) -> bool:
    # Hop le neu la dang universe.roboflow.com/<workspace>/<project>
    if 'universe.roboflow.com' not in url:
        return False
    path_parts = [p for p in urlparse(url).path.split('/') if p]
    if len(path_parts) < 2:
        return False
    # Loai bo trang search va trang chung
    blocked = {'search', 'datasets', 'models', 'projects'}
    return path_parts[0] not in blocked

fallback_candidates = [
    {
        'query': 'fallback',
        'title': 'Roboflow Universe search: helmet',
        'url': 'https://universe.roboflow.com/search?q=helmet',
    },
    {
        'query': 'fallback',
        'title': 'Roboflow Universe search: safety harness',
        'url': 'https://universe.roboflow.com/search?q=safety%20harness',
    },
    {
        'query': 'fallback',
        'title': 'Roboflow Universe search: boots',
        'url': 'https://universe.roboflow.com/search?q=boots',
    },
    {
        'query': 'fallback',
        'title': 'Roboflow Universe search: trash bag',
        'url': 'https://universe.roboflow.com/search?q=trash%20bag',
    },
    {
        'query': 'fallback',
        'title': 'Roboflow Universe search: wet floor sign',
        'url': 'https://universe.roboflow.com/search?q=wet%20floor%20sign',
    },
]

candidates = []
search_errors = []

print('Dang tim dataset moi...')
with DDGS() as ddgs:
    for q in queries:
        try:
            # max_results cao hon de tang co hoi bat duoc URL Roboflow hop le
            results = list(ddgs.text(q, max_results=40))
            print(f'- Query: {q} -> {len(results)} ket qua')
            for r in results:
                link = r.get('href') or r.get('url') or ''
                title = r.get('title', '')
                if not link:
                    continue
                candidates.append({
                    'query': q,
                    'title': title,
                    'url': normalize_url(link),
                })
        except Exception as qe:
            search_errors.append(f'Query error [{q}]: {qe}')

if search_errors:
    print('\nCanh bao search:')
    for err in search_errors[:10]:
        print('-', err)

unique = {}
for c in candidates:
    u = c['url']
    if not u or u in unique:
        continue
    if any(u.startswith(old) for old in OLD_DATASET_URLS):
        continue

    # Uu tien URL project Roboflow Universe hop le
    if is_probably_rf_project(u):
        unique[u] = c
        continue

    # Van giu mot so URL dataset/document de tham khao
    if 'dataset' in u.lower() or 'detect' in u.lower() or 'github.com' in u.lower():
        unique[u] = c

filtered = list(unique.values())

# Neu ket qua it, tu dong them fallback de ban co duong link tim thu cong ngay
if len(filtered) < 8:
    print('\nKet qua tu dong con it, bo sung fallback candidates de ban chon nhanh.')
    existing_urls = {item['url'] for item in filtered}
    for fb in fallback_candidates:
        if fb['url'] not in existing_urls:
            filtered.append(fb)

print(f"\nTim thay {len(filtered)} URL goi y (da loai URL cu).")
for i, item in enumerate(filtered[:40], 1):
    print(f"{i:02d}. {item.get('title','(no-title)')[:80]}\n    {item.get('url','')}")

(ROOT / 'dataset_candidates_new.json').write_text(
    json.dumps(filtered, ensure_ascii=False, indent=2),
    encoding='utf-8'
 )
print('\nDa luu: dataset_candidates_new.json')

Dang tim dataset moi...
- Query: roboflow universe helmet ppe dataset -> 40 ket qua
- Query: roboflow universe safety harness dataset -> 30 ket qua
- Query: roboflow universe boots detection dataset -> 30 ket qua
- Query: roboflow universe trash bag detection dataset -> 12 ket qua
- Query: roboflow universe wet floor sign dataset -> 10 ket qua
- Query: construction ppe dataset yolo -> 18 ket qua

Tim thay 80 URL goi y (da loai URL cu).
01. DatasetPPE-RoboflowUniverse
    https://universe.roboflow.com/dataset-ppe
02. PPE Datasets and Pre-Trained Models
    https://universe.roboflow.com/browse/manufacturing/ppe
03. keremberke/protective-equipment-detection · Datasets at Hugging Face
    https://huggingface.co/datasets/keremberke/protective-equipment-detection
04. PPE Detection Dataset - Roboflow Universe
    https://universe.roboflow.com/ppe-detection-dataset
05. GitHub - kredd10/SAS25Dataset: PPE dataset for workplace ...
    https://github.com/kredd10/SAS25Dataset
06. Dataset of Person

## Chon Dataset Moi (Khong Dung 5 Bo Cu)

Cell nay la markdown nen KHONG co output khi bam Run.

Ban lam theo thu tu sau:
1) Chay Cell 7 de tim URL goi y va tao file `dataset_candidates_new.json`.
2) (Tuy chon) Chay Cell 9A de xem nhanh co bao nhieu URL hop le.
3) Sua Cell 9 (`SELECTED_DATASETS`) bang link ban chon.
4) Chay Cell 9 roi chay tiep Cell 10, 11...

Format URL Roboflow Universe mong doi: `https://universe.roboflow.com/<workspace>/<project>`

In [5]:
# Cell 9A (quick check): xem ket qua search truoc khi dien SELECTED_DATASETS
import json
from pathlib import Path

candidate_file = Path.cwd() / 'dataset_candidates_new.json'
if not candidate_file.exists():
    print('Chua co dataset_candidates_new.json. Hay chay Cell 7 truoc.')
else:
    items = json.loads(candidate_file.read_text(encoding='utf-8'))
    print(f'Tong URL goi y: {len(items)}')
    for i, it in enumerate(items[:15], 1):
        print(f"{i:02d}. {it.get('url','')}")

Tong URL goi y: 80
01. https://universe.roboflow.com/dataset-ppe
02. https://universe.roboflow.com/browse/manufacturing/ppe
03. https://huggingface.co/datasets/keremberke/protective-equipment-detection
04. https://universe.roboflow.com/ppe-detection-dataset
05. https://github.com/kredd10/SAS25Dataset
06. https://data.mendeley.com/datasets/zkzghjvpn2/1
07. https://universe.roboflow.com/ppe-0rs0f/ppe-dataset-rblaa
08. https://universe.roboflow.com/personal-protective-equipment-mtl62/ppe-zykvq
09. https://universe.roboflow.com/ppe-la0vn/ppe-dataset-for-workplace-safety-qobrx
10. https://universe.roboflow.com/ai-project-yolo/ppe-detection-q897z
11. https://universe.roboflow.com/siabar/ppe-dataset-for-workplace-safety
12. https://universe.roboflow.com/salo-levy-nlqrn/hard-hat-ppe
13. https://universe.roboflow.com/mo-c5bka/ppe-detection-helmet
14. https://universe.roboflow.com/trialforobjectdetection/ppe-v1.1
15. https://universe.roboflow.com/latest-version/new-ppe


In [6]:
# Cell 9B: Kiem tra link Roboflow co quyen export/download hay khong (truoc khi tai that)
from urllib.parse import urlparse
import requests
from dotenv import load_dotenv
from roboflow import Roboflow

# Fallback de Cell 9B co the chay doc lap khi Run All/chay le
if 'SELECTED_DATASETS' not in globals():
    SELECTED_DATASETS = [
        'https://universe.roboflow.com/data-sets-gywm3/trash-bag-detection-os4tb',
        'https://universe.roboflow.com/study-dopms/safety-harness-for-smart-construction',
        'https://universe.roboflow.com/june-2023-wet-floor-sign/wet-floor-sign',
        'https://docs.ultralytics.com/datasets/detect/construction-ppe/',
    ]
    print('[INFO] SELECTED_DATASETS chua duoc khai bao, dang dung danh sach mac dinh.')

if 'ULTRA_CONSTRUCTION_DOC' not in globals():
    ULTRA_CONSTRUCTION_DOC = 'https://docs.ultralytics.com/datasets/detect/construction-ppe/'
if 'ROBOFLOW_API_URL' not in globals():
    ROBOFLOW_API_URL = 'https://api.roboflow.com'

if 'parse_rf_universe_url' not in globals():
    def parse_rf_universe_url(url: str):
        p = urlparse(url)
        parts = [x for x in p.path.split('/') if x]
        if len(parts) < 2:
            raise ValueError(f'URL khong dung format universe: {url}')
        return parts[0], parts[1]

if '_extract_latest_version_number' not in globals():
    def _extract_latest_version_number(project):
        versions = project.versions()
        if not versions:
            raise RuntimeError('Project khong co version nao.')
        nums = []
        for v in versions:
            if hasattr(v, 'version'):
                nums.append(int(getattr(v, 'version')))
            elif isinstance(v, dict):
                if 'version' in v:
                    nums.append(int(v['version']))
                elif 'id' in v:
                    nums.append(int(v['id']))
        if not nums:
            raise RuntimeError('Khong doc duoc so version tu project.versions().')
        return max(nums)

if 'api_key' not in globals() or not globals().get('api_key') or 'rf' not in globals():
    load_dotenv()
    api_key = os.getenv('ROBOFLOW_API_KEY')
    if not api_key:
        raise ValueError('Thieu ROBOFLOW_API_KEY trong .env')
    rf = Roboflow(api_key=api_key)

def validate_rf_downloadable(url: str):
    url = url.rstrip('/')
    if 'universe.roboflow.com' not in url:
        return {'url': url, 'ok': True, 'reason': 'non-roboflow (skip validate)'}
    try:
        ws, pj = parse_rf_universe_url(url)
        project = rf.workspace(ws).project(pj)
        latest = _extract_latest_version_number(project)

        # Kiem tra endpoint export API
        api_url = f"{ROBOFLOW_API_URL}/{ws}/{pj}/{latest}/yolov8?api_key={api_key}&nocache=true"
        api_resp = requests.get(api_url, timeout=30)
        if api_resp.status_code >= 400:
            return {'url': url, 'ok': False, 'reason': f'api status {api_resp.status_code} at v{latest}'}

        payload = api_resp.json()
        export_link = payload.get('export', {}).get('link', '')
        if not export_link:
            return {'url': url, 'ok': False, 'reason': f'khong co export.link tai v{latest}'}

        # Kiem tra nhanh link tai zip (khong in api key)
        z = requests.get(export_link, timeout=30, allow_redirects=True)
        if z.status_code >= 400:
            return {'url': url, 'ok': False, 'reason': f'export link status {z.status_code} at v{latest}'}

        return {'url': url, 'ok': True, 'reason': f'downloadable via export.link at v{latest}'}
    except Exception as e:
        return {'url': url, 'ok': False, 'reason': str(e)}

print('=== Validate SELECTED_DATASETS truoc khi tai ===')
validation_results = []
for u in SELECTED_DATASETS:
    if u.rstrip('/') == ULTRA_CONSTRUCTION_DOC.rstrip('/'):
        validation_results.append({'url': u, 'ok': True, 'reason': 'Ultralytics dataset'})
        continue
    validation_results.append(validate_rf_downloadable(u))

for r in validation_results:
    status = 'OK' if r['ok'] else 'FAIL'
    print(f"[{status}] {r['url']} -> {r['reason']}")

valid_selected = [r['url'] for r in validation_results if r['ok']]
invalid_selected = [r['url'] for r in validation_results if not r['ok']]

print('\nCo the tai duoc:', len(valid_selected))
for u in valid_selected:
    print('-', u)

if invalid_selected:
    print('\nCan doi link (khong tai duoc):')
    for u in invalid_selected:
        print('-', u)

[INFO] SELECTED_DATASETS chua duoc khai bao, dang dung danh sach mac dinh.
=== Validate SELECTED_DATASETS truoc khi tai ===
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
[OK] https://universe.roboflow.com/data-sets-gywm3/trash-bag-detection-os4tb -> downloadable via export.link at v1
[OK] https://universe.roboflow.com/study-dopms/safety-harness-for-smart-construction -> downloadable via export.link at v3
[OK] https://universe.roboflow.com/june-2023-wet-floor-sign/wet-floor-sign -> downloadable via export.link at v1
[OK] https://docs.ultralytics.com/datasets/detect/construction-ppe/ -> Ultralytics dataset

Co the tai duoc: 4
- https://universe.roboflow.com/data-sets-gywm3/trash-bag-detection-os4tb
- https://universe.roboflow.com/study-dopms/safety-harness-for-smart-construction
- https://universe.roboflow.com/june-2023-wet-floor-sign/wet-floor-sign
- https://do

In [7]:
# Ban da chon 4 dataset nay
SELECTED_DATASETS = [
    'https://universe.roboflow.com/data-sets-gywm3/trash-bag-detection-os4tb',
    'https://universe.roboflow.com/study-dopms/safety-harness-for-smart-construction',
    'https://universe.roboflow.com/june-2023-wet-floor-sign/wet-floor-sign',
    'https://docs.ultralytics.com/datasets/detect/construction-ppe/',
]

assert len(SELECTED_DATASETS) > 0, 'Hay them it nhat 1 dataset moi vao SELECTED_DATASETS'
print('Da chon', len(SELECTED_DATASETS), 'dataset')
for i, u in enumerate(SELECTED_DATASETS, 1):
    print(f'{i}. {u}')

Da chon 4 dataset
1. https://universe.roboflow.com/data-sets-gywm3/trash-bag-detection-os4tb
2. https://universe.roboflow.com/study-dopms/safety-harness-for-smart-construction
3. https://universe.roboflow.com/june-2023-wet-floor-sign/wet-floor-sign
4. https://docs.ultralytics.com/datasets/detect/construction-ppe/


In [8]:
from dotenv import load_dotenv
from roboflow import Roboflow
from zipfile import ZipFile
import requests

load_dotenv()
api_key = os.getenv('ROBOFLOW_API_KEY')
if not api_key:
    raise ValueError('Thieu ROBOFLOW_API_KEY trong .env')

rf = Roboflow(api_key=api_key)

ULTRA_CONSTRUCTION_DOC = 'https://docs.ultralytics.com/datasets/detect/construction-ppe/'
ULTRA_CONSTRUCTION_ZIP = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip'
ROBOFLOW_API_URL = 'https://api.roboflow.com'

def parse_rf_universe_url(url: str):
    p = urlparse(url)
    parts = [x for x in p.path.split('/') if x]
    if len(parts) < 2:
        raise ValueError(f'URL khong dung format universe: {url}')
    return parts[0], parts[1]

def find_dataset_root(start_dir: Path):
    cands = [start_dir] + [p.parent for p in start_dir.rglob('data.yaml')] + [p.parent for p in start_dir.rglob('construction-ppe.yaml')]
    seen = set()
    for c in cands:
        c = Path(c)
        if c in seen:
            continue
        seen.add(c)
        if (c / 'data.yaml').exists() or (c / 'construction-ppe.yaml').exists():
            return c
    return start_dir

def _extract_latest_version_number(project):
    versions = project.versions()
    if not versions:
        raise RuntimeError('Project khong co version nao.')
    nums = []
    for v in versions:
        if hasattr(v, 'version'):
            nums.append(int(getattr(v, 'version')))
        elif isinstance(v, dict):
            if 'version' in v:
                nums.append(int(v['version']))
            elif 'id' in v:
                nums.append(int(v['id']))
    if not nums:
        raise RuntimeError('Khong doc duoc so version tu project.versions().')
    return max(nums)

def _get_export_link_from_api(workspace: str, project_name: str, version: int, fmt: str = 'yolov8'):
    """Lay link export chinh thuc tu api.roboflow.com."""
    url = f"{ROBOFLOW_API_URL}/{workspace}/{project_name}/{version}/{fmt}?api_key={api_key}&nocache=true"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    payload = resp.json()
    export = payload.get('export', {})
    link = export.get('link')
    if not link:
        raise RuntimeError(f'Khong co export.link cho {workspace}/{project_name} v{version} format={fmt}')
    return link

def _download_from_export_link(export_link: str, out_dir: Path, project_name: str, version: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir / f'{project_name}-v{version}.zip'
    r = requests.get(export_link, timeout=120, allow_redirects=True)
    r.raise_for_status()
    zip_path.write_bytes(r.content)
    with ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    return find_dataset_root(out_dir)

def _download_rf_direct(workspace: str, project_name: str, out_dir: Path, version: int):
    """Fallback robust: lay export.link tu API roi tai zip tu link do."""
    export_link = _get_export_link_from_api(workspace, project_name, version, fmt='yolov8')
    return _download_from_export_link(export_link, out_dir, project_name, version)

def download_latest_yolov8(workspace: str, project_name: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    project = rf.workspace(workspace).project(project_name)

    latest_version_num = _extract_latest_version_number(project)
    print(f'Latest version cua {workspace}/{project_name}: v{latest_version_num}')

    try:
        print(f'Thu SDK download {workspace}/{project_name} v{latest_version_num} ...')
        project.version(latest_version_num).download('yolov8', location=str(out_dir))
        root = find_dataset_root(out_dir)
        if (root / 'data.yaml').exists() or (root / 'construction-ppe.yaml').exists():
            print(f'[OK][SDK] {workspace}/{project_name} v{latest_version_num} -> {root}')
            return root
        raise RuntimeError('SDK tai xong nhung khong tim thay file yaml dataset.')
    except Exception as sdk_err:
        print(f'[WARN] SDK loi: {sdk_err}')

    try:
        print(f'Thu API export-link download {workspace}/{project_name} v{latest_version_num} ...')
        root = _download_rf_direct(workspace, project_name, out_dir, latest_version_num)
        if (root / 'data.yaml').exists() or (root / 'construction-ppe.yaml').exists():
            print(f'[OK][API-LINK] {workspace}/{project_name} v{latest_version_num} -> {root}')
            return root
        raise RuntimeError('Tai zip xong nhung khong tim thay file yaml dataset.')
    except requests.HTTPError as direct_http:
        if getattr(direct_http.response, 'status_code', None) == 403:
            raise RuntimeError(
                f'403 forbidden: dataset {workspace}/{project_name} khong cho phep export/download bang API key hien tai.'
)
        raise RuntimeError(
            f'Khong download duoc dataset {workspace}/{project_name} (latest v{latest_version_num}). Loi HTTP: {direct_http}'
)
    except Exception as direct_err:
        raise RuntimeError(
            f'Khong download duoc dataset {workspace}/{project_name} (latest v{latest_version_num}). Loi: {direct_err}'
)

def download_ultralytics_construction_ppe(out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir / 'construction-ppe.zip'
    print('[INFO] Download construction-ppe tu Ultralytics assets...')
    r = requests.get(ULTRA_CONSTRUCTION_ZIP, timeout=120)
    r.raise_for_status()
    zip_path.write_bytes(r.content)

    with ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)

    ds_root = find_dataset_root(out_dir)
    if not ((ds_root / 'data.yaml').exists() or (ds_root / 'construction-ppe.yaml').exists()):
        raise RuntimeError('Khong tim thay data.yaml sau khi giai nen construction-ppe')

    print(f'[OK] Ultralytics construction-ppe tai: {ds_root}')
    return ds_root

In [9]:
downloaded_dirs = []
failed_datasets = []

for url in SELECTED_DATASETS:
    url_norm = url.rstrip('/')
    try:
        # Nhan nhanh dataset construction-ppe tu Ultralytics
        if url_norm == ULTRA_CONSTRUCTION_DOC.rstrip('/'):
            dst = DATA_ROOT / 'ultralytics__construction_ppe'
            if (dst / 'data.yaml').exists() or (dst / 'construction-ppe.yaml').exists() or list(dst.rglob('data.yaml')) or list(dst.rglob('construction-ppe.yaml')):
                print(f'[SKIP] Da co san Ultralytics dataset: {dst}')
                ds_root = find_dataset_root(dst)
            else:
                ds_root = download_ultralytics_construction_ppe(dst)
            downloaded_dirs.append(ds_root)
            continue

        # Nhan URL Roboflow Universe
        ws, pj = parse_rf_universe_url(url_norm)
        dst = DATA_ROOT / f'{ws}__{pj}'
        ds_root = download_latest_yolov8(ws, pj, dst)
        downloaded_dirs.append(ds_root)
    except Exception as e:
        failed_datasets.append({'url': url_norm, 'error': str(e)})
        print(f'[FAIL] {url_norm} -> {e}')

# verify sau download
verified_dirs = []
for d in downloaded_dirs:
    root = find_dataset_root(d)
    has_yaml = (root / 'data.yaml').exists() or (root / 'construction-ppe.yaml').exists()
    print(f'[VERIFY] {root} | yaml={has_yaml}')
    if has_yaml:
        verified_dirs.append(root)
    else:
        failed_datasets.append({'url': str(d), 'error': 'Khong tim thay data.yaml'})

downloaded_dirs = verified_dirs
print('\nTong dataset tai thanh cong:', len(downloaded_dirs))
for d in downloaded_dirs:
    print('-', d)

if failed_datasets:
    print('\nDataset bi loi/can doi link:')
    for item in failed_datasets:
        print('-', item['url'])
        print('  error:', item['error'])

if len(downloaded_dirs) == 0:
    raise RuntimeError('Khong dataset nao tai thanh cong. Can doi link Roboflow (uu tien link co dataset version cong khai).')

loading Roboflow workspace...
loading Roboflow project...
Latest version cua data-sets-gywm3/trash-bag-detection-os4tb: v1
Thu SDK download data-sets-gywm3/trash-bag-detection-os4tb v1 ...
[OK][SDK] data-sets-gywm3/trash-bag-detection-os4tb v1 -> e:\capstone\cleanopsai-ai\new_data_sources\data-sets-gywm3__trash-bag-detection-os4tb
loading Roboflow workspace...
loading Roboflow project...
Latest version cua study-dopms/safety-harness-for-smart-construction: v3
Thu SDK download study-dopms/safety-harness-for-smart-construction v3 ...
[OK][SDK] study-dopms/safety-harness-for-smart-construction v3 -> e:\capstone\cleanopsai-ai\new_data_sources\study-dopms__safety-harness-for-smart-construction
loading Roboflow workspace...
loading Roboflow project...
Latest version cua june-2023-wet-floor-sign/wet-floor-sign: v1
Thu SDK download june-2023-wet-floor-sign/wet-floor-sign v1 ...
[OK][SDK] june-2023-wet-floor-sign/wet-floor-sign v1 -> e:\capstone\cleanopsai-ai\new_data_sources\june-2023-wet-floo

In [10]:
# Cell 13A: Kiem toan mapping class truoc khi merge (standalone)
_NEGATIVE_HINTS_LOCAL = ['no_', 'without', 'none', 'background']
_TARGET_MAPPING_LOCAL = {
    'glove': 0, 'gloves': 0,
    'mask': 1, 'surgical_mask': 1, 'n95': 1, 'medical_mask': 1,
    'goggle': 2, 'goggles': 2, 'face_shield': 2, 'shield': 2, 'visor': 2,
    'coverall': 3, 'biohazard_suit': 3, 'hazmat': 3, 'protective_suit': 3,
    'helmet': 4, 'hardhat': 4, 'hard_hat': 4, 'hard hat': 4,
    'vest': 5, 'safety_vest': 5, 'reflective_vest': 5,
    'harness': 6, 'safety_harness': 6, 'rope': 6, 'lifeline': 6,
    'boot': 7, 'boots': 7, 'shoe': 7, 'shoes': 7, 'safety_shoes': 7,
    'trash_bag': 8, 'trashbag': 8, 'garbage_bag': 8, 'bag': 8, 'waste_bag': 8,
    'wet_floor_sign': 9, 'wet floor sign': 9, 'wetfloorsign': 9, 'wet_floor': 9, 'slippery_sign': 9, 'caution_sign': 9, 'warning_sign': 9,
}

def _normalize_name_local(name):
    return re.sub(r'[^a-z0-9_ ]+', '', str(name).lower().strip())

def _load_names_from_yaml(ds_root: Path):
    y = ds_root / 'data.yaml'
    if not y.exists():
        y = ds_root / 'construction-ppe.yaml'
    info = yaml.safe_load(y.read_text(encoding='utf-8')) or {}
    names = info.get('names', [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]
    return names

def _preview_mapping(names_array):
    mapped = {}
    unmapped = []
    for local_id, class_name in enumerate(names_array):
        clean_name = _normalize_name_local(str(class_name))
        if any(k in clean_name for k in _NEGATIVE_HINTS_LOCAL):
            continue
        matched = None
        for key, master_id in _TARGET_MAPPING_LOCAL.items():
            if key in clean_name:
                matched = master_id
                break
        if matched is None:
            unmapped.append((local_id, class_name))
        else:
            mapped[local_id] = matched
    return mapped, unmapped

print('=== Mapping audit truoc merge ===')
if 'failed_datasets' in globals() and failed_datasets:
    print('Can xem lai link loi truoc khi train full:')
    for item in failed_datasets:
        print('-', item['url'])

for ds in downloaded_dirs:
    names = _load_names_from_yaml(ds)
    mapped, unmapped = _preview_mapping(names)
    print(f'\n[{ds.name}]')
    print('names:', names)
    print('mapped:', mapped)
    print('unmapped:', unmapped if unmapped else 'none')

=== Mapping audit truoc merge ===

[data-sets-gywm3__trash-bag-detection-os4tb]
names: ['Trash Bag']
mapped: {0: 8}
unmapped: none

[study-dopms__safety-harness-for-smart-construction]
names: ['harness', 'harness_bad', 'harness_good', 'no_harness']
mapped: {0: 6, 1: 6, 2: 6}
unmapped: none

[june-2023-wet-floor-sign__wet-floor-sign]
names: ['Wet-Floor-Sign']
mapped: {0: 9}
unmapped: none

[ultralytics__construction_ppe]
names: ['helmet', 'gloves', 'vest', 'boots', 'goggles', 'none', 'Person', 'no_helmet', 'no_goggle', 'no_gloves', 'no_boots']
mapped: {0: 4, 1: 0, 2: 5, 3: 7, 4: 2}
unmapped: [(6, 'Person')]


## Remap Ve 10 Class Chuan + Gop Dataset

Cell nay tu dong remap nhan dong nghia ve 10 class muc tieu.
Co bo sung nhieu tu dong nghia cho 5 class dang yeu de tang do bao phu du lieu.

In [11]:
for split in ['train', 'valid']:
    (MERGED_ROOT / f'images/{split}').mkdir(parents=True, exist_ok=True)
    (MERGED_ROOT / f'labels/{split}').mkdir(parents=True, exist_ok=True)

target_mapping = {
    'glove': 0, 'gloves': 0,
    'mask': 1, 'surgical_mask': 1, 'n95': 1, 'medical_mask': 1,
    'goggle': 2, 'goggles': 2, 'face_shield': 2, 'shield': 2, 'visor': 2,
    'coverall': 3, 'biohazard_suit': 3, 'hazmat': 3, 'protective_suit': 3,
    'helmet': 4, 'hardhat': 4, 'hard_hat': 4, 'hard hat': 4,
    'vest': 5, 'safety_vest': 5, 'reflective_vest': 5,
    'harness': 6, 'safety_harness': 6, 'rope': 6, 'lifeline': 6,
    'boot': 7, 'boots': 7, 'shoe': 7, 'shoes': 7, 'safety_shoes': 7,
    'trash_bag': 8, 'trashbag': 8, 'garbage_bag': 8, 'bag': 8, 'waste_bag': 8,
    'wet_floor_sign': 9, 'wet floor sign': 9, 'wetfloorsign': 9, 'wet_floor': 9, 'slippery_sign': 9, 'caution_sign': 9, 'warning_sign': 9,
}

NEGATIVE_HINTS = ['no_', 'without', 'none', 'background']

def normalize_name(name):
    return re.sub(r'[^a-z0-9_ ]+', '', name.lower().strip())

def build_local_to_master(names_array):
    local_to_master = {}
    for local_id, class_name in enumerate(names_array):
        clean_name = normalize_name(str(class_name))
        if any(k in clean_name for k in NEGATIVE_HINTS):
            continue

        matched = None
        for key, master_id in target_mapping.items():
            if key in clean_name:
                matched = master_id
                break
        if matched is not None:
            local_to_master[local_id] = matched
    return local_to_master

def _resolve_split_path(root: Path, yaml_path: Path, split_value: str):
    p = Path(str(split_value))
    if p.is_absolute():
        return p
    # uu tien relative theo truong `path` trong yaml
    info = yaml.safe_load(yaml_path.read_text(encoding='utf-8')) or {}
    base = Path(info.get('path', '.'))
    if not base.is_absolute():
        base = (yaml_path.parent / base).resolve()
    cand1 = (base / p).resolve()
    if cand1.exists():
        return cand1
    cand2 = (root / p).resolve()
    if cand2.exists():
        return cand2
    return cand1

def _to_img_lbl_dirs(candidate: Path):
    candidate = Path(candidate)
    # candidate la .../images
    if candidate.name.lower() == 'images':
        return candidate, candidate.parent / 'labels'
    # candidate la .../images/train
    if candidate.parent.name.lower() == 'images':
        return candidate, candidate.parent.parent / 'labels' / candidate.name
    # candidate la .../train
    return candidate / 'images', candidate / 'labels'

def _find_split_dirs(source_dir: Path, yaml_path: Path, split_key: str, split_value: str):
    raw = Path(str(split_value))
    resolved = _resolve_split_path(source_dir, yaml_path, split_value)

    candidates = [resolved]

    # Fallback cho Roboflow: bo '..' de tro ve root dataset hien tai
    cleaned_parts = [part for part in raw.parts if part not in ('..', '.')]
    if cleaned_parts:
        candidates.append((source_dir / Path(*cleaned_parts)).resolve())

    key_aliases = [split_key]
    if split_key == 'val':
        key_aliases.append('valid')
    if split_key == 'valid':
        key_aliases.append('val')

    for alias in key_aliases:
        candidates.append((source_dir / alias).resolve())
        candidates.append((source_dir / alias / 'images').resolve())
        candidates.append((source_dir / 'images' / alias).resolve())

    seen = set()
    for c in candidates:
        c = Path(c)
        if c in seen:
            continue
        seen.add(c)
        img_dir, lbl_dir = _to_img_lbl_dirs(c)
        if img_dir.exists() and lbl_dir.exists():
            return img_dir, lbl_dir, c

    # tra ve cap tu candidate dau tien de in warning de debug
    img_dir, lbl_dir = _to_img_lbl_dirs(candidates[0])
    return img_dir, lbl_dir, candidates[0]

def process_one_dataset(source_dir: Path, prefix: str):
    # Ho tro ca data.yaml (Roboflow) va construction-ppe.yaml (Ultralytics)
    yaml_path = source_dir / 'data.yaml'
    if not yaml_path.exists():
        alt_yaml = source_dir / 'construction-ppe.yaml'
        yaml_path = alt_yaml if alt_yaml.exists() else yaml_path
    if not yaml_path.exists():
        print(f'[SKIP] Khong thay file yaml dataset: {source_dir}')
        return defaultdict(int)

    with open(yaml_path, 'r', encoding='utf-8') as f:
        data_info = yaml.safe_load(f) or {}

    names_array = data_info.get('names', [])
    if isinstance(names_array, dict):
        names_array = [names_array[k] for k in sorted(names_array.keys())]

    local_to_master = build_local_to_master(names_array)
    print(f'[{prefix}] local->master:', local_to_master)

    stats = defaultdict(int)

    split_keys = [('train', 'train'), ('val', 'valid'), ('valid', 'valid'), ('test', 'valid')]
    for key, target_split in split_keys:
        if key not in data_info:
            continue

        img_dir, lbl_dir, used_base = _find_split_dirs(source_dir, yaml_path, key, data_info[key])
        if not img_dir.exists() or not lbl_dir.exists():
            print(f'[WARN] Khong thay images/labels cho split={key}: {used_base}')
            continue

        for txt_file in lbl_dir.glob('*.txt'):
            lines = txt_file.read_text(encoding='utf-8').splitlines()
            new_lines = []

            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                old_id = int(parts[0])
                if old_id in local_to_master:
                    new_id = local_to_master[old_id]
                    new_lines.append(f"{new_id} {' '.join(parts[1:])}")
                    stats[TARGET_NAMES[new_id]] += 1

            if not new_lines:
                continue

            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.webp', '.JPG', '.JPEG', '.PNG', '.WEBP']:
                candidate = img_dir / (txt_file.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if img_path is None:
                continue

            dest_img = MERGED_ROOT / f'images/{target_split}' / f'{prefix}_{img_path.name}'
            dest_lbl = MERGED_ROOT / f'labels/{target_split}' / f'{prefix}_{txt_file.name}'

            shutil.copy2(img_path, dest_img)
            dest_lbl.write_text('\n'.join(new_lines) + '\n', encoding='utf-8')

    return stats

In [12]:
all_stats = defaultdict(int)
for ds in downloaded_dirs:
    prefix = ds.name.replace('-', '_')
    st = process_one_dataset(ds, prefix)
    for k, v in st.items():
        all_stats[k] += v

print('\n=== Label counts sau khi gop ===')
for i in range(10):
    name = TARGET_NAMES[i]
    print(f'{name:18s}: {all_stats.get(name, 0)}')

train_imgs = len(list((MERGED_ROOT / 'images/train').glob('*.*')))
valid_imgs = len(list((MERGED_ROOT / 'images/valid').glob('*.*')))
print('\nTrain images:', train_imgs)
print('Valid images:', valid_imgs)

[data_sets_gywm3__trash_bag_detection_os4tb] local->master: {0: 8}
[WARN] Khong thay images/labels cho split=val: E:\capstone\cleanopsai-ai\new_data_sources\valid\images
[WARN] Khong thay images/labels cho split=test: E:\capstone\cleanopsai-ai\new_data_sources\test\images
[study_dopms__safety_harness_for_smart_construction] local->master: {0: 6, 1: 6, 2: 6}
[june_2023_wet_floor_sign__wet_floor_sign] local->master: {0: 9}
[WARN] Khong thay images/labels cho split=val: E:\capstone\cleanopsai-ai\new_data_sources\valid\images
[ultralytics__construction_ppe] local->master: {0: 4, 1: 0, 2: 5, 3: 7, 4: 2}

=== Label counts sau khi gop ===
gloves            : 1461
surgical_mask     : 0
face_shield       : 526
biohazard_suit    : 0
helmet            : 1750
safety_vest       : 1632
safety_harness    : 107
boots             : 1613
trash_bag         : 74
wet_floor_sign    : 89

Train images: 1087
Valid images: 319


## Tao YAML Moi

YAML dung cho training incremental tren dataset moi da gop.

In [13]:
data_yaml_path = ROOT / 'master_dataset_v2_new_only.yaml'
data_cfg = {
    'path': str(MERGED_ROOT.absolute()),
    'train': 'images/train',
    'val': 'images/valid',
    'names': TARGET_NAMES,
}

with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print('Da tao:', data_yaml_path)

Da tao: e:\capstone\cleanopsai-ai\master_dataset_v2_new_only.yaml


## Incremental Fine-tune (Train Tiep)

Khuyen nghi train tiep tu `best_ppe_model_v1.pt` de nhanh va on dinh hon train lai tu dau.

Goi y tham so:
- learning rate thap (`lr0=3e-4`)
- epoch 40-80
- dung `cos_lr=True`
- giu `imgsz=640` de can bang toc do/chinh xac

In [14]:
resume_ckpt = ROOT / 'best_ppe_model_v1.pt'
if not resume_ckpt.exists():
    raise FileNotFoundError('Khong tim thay best_ppe_model_v1.pt de fine-tune')

# Chan train som neu merge chua co du lieu
train_count = len(list((MERGED_ROOT / 'images/train').glob('*.*')))
valid_count = len(list((MERGED_ROOT / 'images/valid').glob('*.*')))
print(f'Merged train images: {train_count}')
print(f'Merged valid images: {valid_count}')
if train_count == 0:
    raise RuntimeError(
        'Master_Dataset_v2/images/train dang rong. Hay chay lai Cell 12 -> Cell 15 theo thu tu de download + merge du lieu truoc khi train.'
    )

model = YOLO(str(resume_ckpt))

results = model.train(
    data=str(data_yaml_path),
    epochs=60,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project=str(RUNS_ROOT),
    name='ppe_incremental_new_data_v1',
    exist_ok=True,
    optimizer='AdamW',
    lr0=3e-4,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    close_mosaic=10,
    patience=20,
    amp=True,
)

best_src = Path(results.save_dir) / 'weights' / 'best.pt'
best_dst = ROOT / 'best_ppe_model_v2_incremental.pt'
if best_src.exists():
    shutil.copy2(best_src, best_dst)
    print('Da luu model moi:', best_dst)
else:
    print('Khong thay best.pt sau training')

Merged train images: 1087
Merged valid images: 319
New https://pypi.org/project/ultralytics/8.4.27 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=e:\capstone\cleanopsai-ai\master_dataset_v2_new_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

## Danh Gia Nhanh Theo Class

Tap trung check 5 class yeu: helmet, safety_harness, boots, trash_bag, wet_floor_sign.

In [15]:
metrics = model.val(data=str(data_yaml_path), split='val')
print(metrics)

Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 93 layers, 25,845,550 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 456.5157.3 MB/s, size: 56.4 KB)
val: Scanning E:\capstone\cleanopsai-ai\Master_Dataset_v2\labels\valid.cache... 319 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 319/319  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 1580. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.9it/s 10.6s0.5s
                   all        319       1580      0.822      0.742      0.777      0.436
                gloves        143        299      0.861      0.706      0.778      0.361
    

## Giam Nham Lan Mot Cach An Toan (Khong Lam Te Di Model)

Ban dung: loi nay chi thinh thoang xay ra, nen khong nen sua qua manh tay.

Nguyen tac an toan:
- Mac dinh GIU NGUYEN model va threshold hien tai.
- Chi tinh chinh khi A/B test tren tap anh that cua ban cho thay cai thien ro rang.
- Neu chi nham le te, uu tien giai phap nhe: bo sung it hard-negative roi fine-tune ngan 10-15 epochs.
- Khong tang threshold `wet_floor_sign` toan he thong neu chua do duoc recall bi giam.

Muc tieu: giam nham nho nhung khong danh doi kha nang nhan dung `wet_floor_sign` hien dang on.

In [16]:
# Cell bo sung 1 (safe): Quy trinh kiem tra truoc khi sua model

print('=== Quy trinh safe rollout ===')
print('1) Thu thap 50-200 anh thuc te gan voi production.')
print('2) Danh dau rieng cac ca nham lan hiem (helmet/tui rac vat vang -> wet_floor_sign).')
print('3) Chay baseline model hien tai va ghi lai precision/recall cho wet_floor_sign.')
print('4) Neu false positive rat thap (le te), KHONG CAN doi threshold/model.')
print('5) Chi khi can thiet moi bo sung hard-negative va fine-tune ngan 10-15 epochs.')
print('6) A/B test: chi deploy neu recall wet_floor_sign khong giam dang ke.')

# Guardrail goi y (ban co the sua theo muc tieu that):
MAX_ALLOWED_RECALL_DROP = 0.01  # <=1%
MIN_REQUIRED_FP_REDUCTION = 0.10 # >=10% moi dang doi
print('\nGuardrail:')
print('- Recall drop toi da cho phep:', MAX_ALLOWED_RECALL_DROP)
print('- Muc giam FP toi thieu de chap nhan:', MIN_REQUIRED_FP_REDUCTION)

=== Quy trinh safe rollout ===
1) Thu thap 50-200 anh thuc te gan voi production.
2) Danh dau rieng cac ca nham lan hiem (helmet/tui rac vat vang -> wet_floor_sign).
3) Chay baseline model hien tai va ghi lai precision/recall cho wet_floor_sign.
4) Neu false positive rat thap (le te), KHONG CAN doi threshold/model.
5) Chi khi can thiet moi bo sung hard-negative va fine-tune ngan 10-15 epochs.
6) A/B test: chi deploy neu recall wet_floor_sign khong giam dang ke.

Guardrail:
- Recall drop toi da cho phep: 0.01
- Muc giam FP toi thieu de chap nhan: 0.1


In [17]:
# Cell bo sung 2 (safe): Co che threshold rieng NHUNG mac dinh tat

import requests
from PIL import Image
from io import BytesIO

# Mac dinh KHONG bat class-threshold de tranh tac dong toi recall hien dang on
ENABLE_CLASS_SPECIFIC_THRESHOLD = False

CLASS_CONF_THRESH = {
    'wet_floor_sign': 0.55,  # chi dung khi ENABLE_CLASS_SPECIFIC_THRESHOLD=True
}
DEFAULT_CONF = 0.25

def predict_baseline(model, image_url: str):
    r = requests.get(image_url, timeout=30)
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert('RGB')
    results = model(img)
    out = []
    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            name = str(model.names[cls_id]).lower()
            if conf >= DEFAULT_CONF:
                out.append({'name': name, 'confidence': round(conf * 100, 2), 'mode': 'baseline'})
    return out

def predict_with_optional_class_threshold(model, image_url: str):
    r = requests.get(image_url, timeout=30)
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert('RGB')
    results = model(img)
    out = []
    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            name = str(model.names[cls_id]).lower()

            need = DEFAULT_CONF
            if ENABLE_CLASS_SPECIFIC_THRESHOLD:
                need = CLASS_CONF_THRESH.get(name, DEFAULT_CONF)

            if conf >= need:
                out.append({
                    'name': name,
                    'confidence': round(conf * 100, 2),
                    'threshold_used': need,
                    'mode': 'class-threshold' if ENABLE_CLASS_SPECIFIC_THRESHOLD else 'baseline-like',
                })
    return out

TEST_IMAGE_URL = ''  # 'https://...jpg'
if TEST_IMAGE_URL:
    base = predict_baseline(model, TEST_IMAGE_URL)
    test = predict_with_optional_class_threshold(model, TEST_IMAGE_URL)
    print('Baseline detections:')
    for p in base:
        print(p)
    print('\nOptional-threshold detections:')
    for p in test:
        print(p)
    print('\nKhuyen nghi: chi bat ENABLE_CLASS_SPECIFIC_THRESHOLD sau khi A/B test tren nhieu anh.')
else:
    print('Hay set TEST_IMAGE_URL de test. Mac dinh dang o safe mode (khong doi hanh vi model).')

Hay set TEST_IMAGE_URL de test. Mac dinh dang o safe mode (khong doi hanh vi model).
